# 5. Add Tools

**Goal:** Add Tavily web search and yfinance OHLCV.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "agent.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agent_utils.knowledge import load_settings

RUN_LIVE = True
settings = load_settings()
model = OpenAIChat(
    id=settings["openrouter_model"],
    api_key=settings["openrouter_api_key"],
    base_url="https://openrouter.ai/api/v1",
)
print("Ready. Live model calls:", RUN_LIVE)

In [ ]:
async def ask(workshop_agent, question, session_id=None):
    if not RUN_LIVE:
        return "Skipped. Set RUN_LIVE = True for a model call."
    response = await workshop_agent.arun(question, session_id=session_id)
    return response.content

In [ ]:
INSTRUCTIONS = [
    "For company financial questions, search the knowledge base first.",
    "If the knowledge base is not available, then tell the user explicitly that you cannot proceed further. Do extra message or hallunicated answer",
    "For latest figures, use the latest period in the knowledge base and name it.",
    "Do not use web search when the knowledge base answers the question.",
    "Answer PDF questions only from retrieved evidence.",
    "Cite the source filename and physical PDF page.",
    "Say when the evidence is missing.",
]

In [ ]:
from agent_utils.knowledge import create_knowledge
knowledge = create_knowledge()
print("Knowledge ready:", knowledge.name)

In [ ]:
from agent_utils.tools import create_tools, market_ohlcv

student_tools = create_tools()
print([getattr(item, "name", getattr(item, "__name__", "")) for item in student_tools])

tool_agent = Agent(
    name="Tool Agent",
    model=model,
    knowledge=knowledge,
    search_knowledge=True,
    tools=student_tools,
    instructions=INSTRUCTIONS,
    tool_call_limit=6,
    markdown=True,
    telemetry=False,
)

if RUN_LIVE:
    print(await market_ohlcv("HCLTECH", period="5d", exchange="NSE"))

    print("Agent response: \n", await ask(tool_agent, question="Fetch 5 days HCLTECH stock ohlcv", session_id="123"))



## Check

Explain what capability this step added and which earlier limitation it fixes.